# EP04 — Decision Tree vs Decision Forest
**COMPSCI 713 · S1 2024 Q4 · 10 marks**

When is a single decision tree preferable to a decision forest? Point out **two aspects** of the problem or context that make it the better choice.

---

## Full Answer [10 marks]

A **decision forest** (Random Forest, Gradient Boosted Trees) almost always beats a single decision tree on raw accuracy. But a single tree is often the *right* choice for the following reasons:

### Aspect 1 — Interpretability & Explainability

A single shallow decision tree is a **fully human-readable flowchart**. Domain experts, regulators, and stakeholders can follow every decision path without ML knowledge.

**When this matters:**
- **Medical decisions:** a doctor needs to explain to a patient or review board exactly why   a diagnosis was made
- **Legal / credit scoring:** regulators (e.g., GDPR Article 22) may require that automated   decisions be explainable; a forest of 500 trees cannot be explained
- **Safety-critical systems:** engineers need to audit the logic before deployment

A forest with 100+ trees is a black box — you get an aggregate prediction but cannot trace the reasoning for any single case.

### Aspect 2 — Resource & Latency Constraints

A decision forest requires evaluating dozens–hundreds of trees and aggregating predictions:

**When this matters:**
- **Embedded / edge devices:** microcontrollers, IoT sensors with strict memory limits   cannot store hundreds of trees
- **Hard real-time requirements:** a single tree executes in O(depth) time;   a forest multiplies this by N trees
- **Simple, well-defined problems:** if the task is inherently low-complexity   (few features, clear decision boundaries), a shallow tree achieves near-identical accuracy   with a fraction of the complexity

**Other valid answers:** small training data where ensembles overfit equally; when training time is severely constrained; regulatory requirements for model simplicity.

---

## Code Demo — Tree vs Forest: accuracy, speed, and readability

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import time, numpy as np

X, y = load_breast_cancer(return_X_y=True)
features = load_breast_cancer().feature_names

dt = DecisionTreeClassifier(max_depth=3, random_state=42)
rf = RandomForestClassifier(n_estimators=200, random_state=42)

# ── Accuracy ──────────────────────────────────────────────────────────────────
dt_acc = cross_val_score(dt, X, y, cv=5).mean()
rf_acc = cross_val_score(rf, X, y, cv=5).mean()
print(f'Decision Tree  (depth=3)     CV accuracy: {dt_acc:.4f}')
print(f'Random Forest  (200 trees)   CV accuracy: {rf_acc:.4f}')

# ── Inference speed ───────────────────────────────────────────────────────────
dt.fit(X, y); rf.fit(X, y)
N = 10_000
Xr = X[np.random.choice(len(X), N)]

t0 = time.perf_counter(); dt.predict(Xr); dt_time = time.perf_counter()-t0
t0 = time.perf_counter(); rf.predict(Xr); rf_time = time.perf_counter()-t0
print(f'\nInference time for {N} samples:')
print(f'  Decision Tree : {dt_time*1000:.2f} ms')
print(f'  Random Forest : {rf_time*1000:.2f} ms  ({rf_time/dt_time:.1f}x slower)')

# ── Readability ───────────────────────────────────────────────────────────────
print('\n=== Decision Tree — fully readable (depth=3) ===')
print(export_text(dt, feature_names=list(features)))
print(f'=== Random Forest: 200 trees, each up to depth {rf.max_depth} ===')
print('   Cannot be printed — total nodes:', sum(t.tree_.node_count for t in rf.estimators_))